In [ ]:
import cv2
from deepface import DeepFace
import sys

# Step 1: Verify all imports are successful
try:
    print("✓ OpenCV version:", cv2.__version__)
    print("✓ DeepFace imported successfully")
except ImportError as e:
    print("✗ Import Error:", e)
    print("Install missing packages with:")
    print("pip install opencv-python deepface")
    sys.exit(1)

# Step 2: Load face cascade classifier
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    print("✗ Error: Could not load Haar Cascade classifier")
    sys.exit(1)
else:
    print("✓ Haar Cascade classifier loaded successfully")

# Step 3: Access webcam with fallback
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Webcam at index 1 not available, trying index 0...")
    cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("✗ Error: Cannot open any webcam")
    print("Troubleshooting:")
    print("- Check if webcam is connected")
    print("- Check if another application is using the webcam")
    print("- Try restarting your computer")
    sys.exit(1)
else:
    print("✓ Webcam opened successfully")

# Step 4: Set webcam properties for better performance
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("\n" + "="*50)
print("Emotion Detection Started")
print("Press 'q' to quit")
print("="*50 + "\n")

# Step 5: Main emotion detection loop
while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to grab frame")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=4
    )

    emotion = "No Face"

    for (x, y, w, h) in faces:

        face = frame[y:y+h, x:x+w]

        try:
            result = DeepFace.analyze(
                face,
                actions=['emotion'],
                enforce_detection=False
            )

            if isinstance(result, list):
                emotion = result[0]["dominant_emotion"]
            else:
                emotion = result["dominant_emotion"]

            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (255, 0, 255),
                2
            )

            cv2.putText(
                frame,
                emotion,
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 0),
                2
            )

        except Exception as e:
            print("DeepFace Error:", str(e))

    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("\nQuitting emotion detection...")
        break

cap.release()
cv2.destroyAllWindows()
print("✓ Application closed successfully")